In [ ]:
#!pip install -U weaviate-client
#!pip install -U langchain-huggingface
#!pip install python-dotenv langchain langchain_community nbimporter
#!pip install pypdf

# Project Setup

## Overview
This notebook is part of a reproducible GRPO-based fine-tuning pipeline for cybersecurity policy generation.

## Purpose of this setup cell
The first code cell in this notebook:
- loads environment variables from `.env`
- identifies the project root directory automatically
- defines standard folder paths used across the project
- creates required folders if they do not already exist

## Why this matters
This makes the notebook portable across macOS, Windows, and Linux without requiring users to manually edit file paths.

## Expected project folders
- `data/corpus/` → source PDF corpus
- `data/processed/` → intermediate processed data
- `data/sample/` → optional small example data
- `outputs/completions/` → generated completions
- `outputs/rankings/` → ranked outputs
- `outputs/models/` → trained model checkpoints
- `outputs/evaluations/` → evaluation results



## Environment and Path Setup

This section loads the required credentials for Weaviate and defines the project folder structure so the notebook can run on any operating system.

In [ ]:

import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Define PROJECT_ROOT automatically
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

# Define folders
DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = DATA_DIR / "corpus"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
COMPLETIONS_DIR = OUTPUT_DIR / "completions"
RANKINGS_DIR = OUTPUT_DIR / "rankings"
MODELS_DIR = OUTPUT_DIR / "models"
EVAL_DIR = OUTPUT_DIR / "evaluations"

# Create folders if needed
for folder in [
    DATA_DIR, CORPUS_DIR, PROCESSED_DIR,
    OUTPUT_DIR, COMPLETIONS_DIR, RANKINGS_DIR, MODELS_DIR, EVAL_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

# 01 Step 1 — Build the Retrieval Corpus in Weaviate

## Purpose
This notebook prepares the retrieval foundation for the project by reading the source PDF corpus, extracting text, chunking the content, generating embeddings, and storing the chunks in Weaviate.

## Why this step is important
The later stages of the pipeline depend on high-quality retrieval. This notebook creates the searchable knowledge base that will be used to enrich prompts with domain-specific cybersecurity context.

## Inputs
- PDF files stored in `data/corpus/`
- Weaviate credentials from `.env`

## Outputs
- A populated Weaviate collection containing embedded cybersecurity policy chunks

## Main tasks
1. Load environment variables
2. Connect to Weaviate
3. Read PDF files from the corpus folder
4. Split text into chunks
5. Embed and upload chunks into the vector database

## Success criteria
This step is complete when the Weaviate collection is created successfully and the document chunks are stored and searchable.

## Environment and Path Setup

This section loads the required credentials for Weaviate and defines the project folder structure so the notebook can run on any operating system.

In [ ]:
WEAVIATE_URL= os.environ["WEAVIATE_URL"] 
WEAVIATE_API_KEY = os.environ["WEAVIATE_API_KEY"] 

In [ ]:
# step 2: Create a client instance and check the connection
import weaviate
from weaviate.auth import AuthApiKey

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=AuthApiKey(WEAVIATE_API_KEY)
)
if client.is_ready():
    print("Weaviate is connected and ready!")
else:
    print("Weaviate connection failed. Check your URL or API key.")

Weaviate is connected and ready!


In [ ]:
client.collections.delete("cybepolicyhunk")
print("🗑️ Deleted 'cyberpolicychunk' collection.")


🗑️ Deleted 'cyberpolicychunk' collection.


In [ ]:
# Step 3: Create schema SDK (Weaviate >= 4.x)

from weaviate.collections.classes.config import Property, DataType

collection_name = "CyberPolicyChunk"

if not client.collections.exists(collection_name):
    client.collections.create(
        name=collection_name,
        properties=[
            Property(name="text", data_type=DataType.TEXT),
            Property(name="source", data_type=DataType.TEXT),
            Property(name="section", data_type=DataType.TEXT)
        ],
        vectorizer_config=None  # Manual embeddings with MiniLM
    )
    print(f"Collection '{collection_name}' created successfully.")
else:
    print(f"Collection '{collection_name}' already exists.")

Collection 'CyberPolicyChunk' created successfully.


In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Define the directory containing the PDF files
pdf_directory = "/content/drive/MyDrive/Corpus"

# List all PDF files in the specified directory
pdf_paths = [os.path.join(pdf_directory, filename)
             for filename in os.listdir(pdf_directory)
             if filename.endswith('.pdf')]

# Load the content of each PDF using PyPDFLoader
docs = [PyPDFLoader(pdf_path).load() for pdf_path in pdf_paths]

# Flatten the list of documents
docs_list = [item for sublist in docs for item in sublist]

In [ ]:
# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
collection = client.collections.get("CyberPolicyChunk")

upload_count = 0
for doc in doc_splits:
    content = doc.page_content
    embedding = embedding_model.embed_documents([content])[0]

    collection.data.insert(
        properties={"text": content, "source": "Unspecified", "section": "General"},
        vector=embedding
    )
    upload_count += 1

print(f"Successfully uploaded {upload_count} document chunks to Weaviate.")

Successfully uploaded 2045 document chunks to Weaviate.


In [ ]:
# Count the number of documents in the collection
collection = client.collections.get("CyberPolicyChunk")
document_count = collection.aggregate.over_all(total_count=True)
print(f"Number of documents in the collection: {document_count.total_count}")

Number of documents in the collection: 2045
